In [5]:
# Core libraries
import pandas as pd
import numpy as np
import json

#preprocessing
from sklearn import set_config
set_config(display = 'diagram', transform_output= "pandas")

# Model selection
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import cross_val_score,cross_validate, TimeSeriesSplit
import optuna, optuna_dashboard

# import for linear model
from sklearn.preprocessing import OneHotEncoder  # pour ElasticNet uniquement
from sklearn.pipeline import make_pipeline       # pour ElasticNet uniquement
from sklearn.preprocessing import StandardScaler # pour ElasticNet uniquement

#feature selection
from sklearn.inspection import permutation_importance

# models
from sklearn.linear_model import ElasticNet #always like to have a linear model for comparison
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns


# Project config
from src.params import *
from src.utils import *
from src.preprocess.cleaning import *
from src.preprocess.features import *
from src.preprocess.pipeline import *

# Plot configuration
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
import warnings
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
df = pd.read_csv("../data/processed/df_processed.csv")

In [7]:
df.head()

,city,date,temp_min,temp_max,cloud_cover,humidity,precipitation,pressure,wind_speed,wind_direction,...,month_cos,month_sin,dow,is_weekend,lag_avg_14,week_std,lag_1,lag_2,lag_3,lag_7
0,Berlin,2023-05-15,10.73,21.05,20.0,56.0,0.0,1008.0,6.17,30.0,...,-0.866025,0.5,0,0,10.735556,1.920113,15.000000,13.266667,11.466667,9.056667
1,Berlin,2023-05-16,10.62,16.48,100.0,73.0,0.0,1014.0,8.75,290.0,...,-0.866025,0.5,1,0,9.513333,1.658554,10.363333,15.000000,13.266667,10.766667
2,Berlin,2023-05-17,6.92,16.03,20.0,46.0,0.0,1022.0,8.23,310.0,...,-0.866025,0.5,2,0,9.422222,2.428601,7.256667,10.363333,15.000000,12.000000
3,Berlin,2023-05-18,6.22,17.46,75.0,47.0,0.0,1016.0,5.81,4.0,...,-0.866025,0.5,3,0,10.040000,3.092590,6.253333,7.256667,10.363333,10.803333
4,Berlin,2023-05-19,7.53,19.38,20.0,46.0,0.0,1026.0,7.15,80.0,...,-0.866025,0.5,4,0,9.908889,3.551526,5.976667,6.253333,7.256667,11.466667


# I. Baseline

There are several baselines we can use to check our models against. 
1. persistence baseline: y_pred = y_today
2. linear extrapolation: y_pred = y_today + (y_today - y_yesterday)
3. last 3 days average: y_pred 

In order to perform a fair comparison with the models, the baseline score must be calculated using the timeseries splits used for model crossvalidation. 

In [ ]:
#creation time serie split
# val set size: 30 days per city (180 because 6 cities)
# gap between train and val set to avoid trailinf effects: 7 days (42 in total)
tscv = TimeSeriesSplit(n_splits= 5, gap= 42, test_size= 180) # val set of 1month per city #one week gap

score_baselines = {"persistence": [],
                   "extrapolation": [],
                   "average": []}

# feature/target split
df = df.sort_values(by="date", ascending= True)
y = df.target
X = df.drop(columns= ["target"])


for train_idx, val_idx in tscv.split(X):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    y_true = y_val
    #persistence baseline
    y_pred = X_val["lag_1"]
    score = root_mean_squared_log_error(np.expm1(y_true), y_pred)
    score_baselines["persistence"].append(score)

    #extrapolation_baseline
    y_pred = (X_val["lag_1"] + (X_val["lag_1"] - X_val["lag_2"])).clip(lower= 0.001)
    score = root_mean_squared_log_error(np.expm1(y_true), y_pred)
    score_baselines["extrapolation"].append(score)

    #average baseline
    y_pred = X_val[["lag_1", "lag_2", "lag_3"]].mean(axis= 1)
    score = root_mean_squared_log_error(np.expm1(y_true), y_pred)
    score_baselines["average"].append(score)


In [ ]:
print("mean score for the three baseline after crossvalidation split:")
print(f"Persistence baseline: {sum(score_baselines["persistence"])/len(score_baselines["persistence"]): .2f}")
print(f"Extrapolation baseline: {sum(score_baselines["extrapolation"])/len(score_baselines["extrapolation"]): .2f}")
print(f"Average baseline: {sum(score_baselines["average"])/len(score_baselines["average"]): .2f}")

mean score for the three baseline after crossvalidation split:
Persistence baseline:  13.61
Extrapolation baseline:  16.87
Average baseline:  12.96
